In [1]:
!pip install -q albumentations tabulate segmentation-models-pytorch kagglehub google

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.0 MB/s eta 0:00:00


In [2]:
!git clone https://github.com/zakariaaithssain/SegKD.git
%cd SegKD

Cloning into 'SegKD'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 36 (delta 11), reused 32 (delta 7), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 1.44 MiB | 25.49 MiB/s, done.
Resolving deltas: 100% (11/11), done.
/content/SegKD


In [3]:
#mount google drive to save the model
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
#download data via kagglehub
import kagglehub
path = kagglehub.dataset_download("lakshaymiddha/crack-segmentation-dataset") + "/crack_segmentation_dataset"
print("Path to dataset files:", path)

100%|██████████| 1.98G/1.98G [00:17<00:00, 119MB/s] 

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/lakshaymiddha/crack-segmentation-dataset/versions/1/crack_segmentation_dataset


In [5]:
import os, shutil, random

src_images = f'{path}/images'
src_masks  = f'{path}/masks'

random.seed(42)
files = sorted(os.listdir(src_images))
random.shuffle(files)

n       = len(files)
n_train = int(n * 0.70)
n_val   = int(n * 0.15)

splits = {
    'train' : files[:n_train],
    'val'   : files[n_train:n_train + n_val],
    'test'  : files[n_train + n_val:],
}

for split, subset in splits.items():
    os.makedirs(f'data/{split}/images', exist_ok=True)
    os.makedirs(f'data/{split}/masks',  exist_ok=True)
    for f in subset:
        mask_f = f if os.path.exists(f'{src_masks}/{f}') else f.replace('.jpg', '.png')
        shutil.copy(f'{src_images}/{f}',     f'data/{split}/images/{f}')
        shutil.copy(f'{src_masks}/{mask_f}', f'data/{split}/masks/{mask_f}')
    print(f'{split}: {len(subset)} images')

train: 7908 images
val: 1694 images
test: 1696 images


In [6]:
!python src/dataset.py

[train] 7908 images chargées
[val] 1694 images chargées
[test] 1696 images chargées
Batch images : torch.Size([4, 3, 512, 512])
Batch masques : torch.Size([4, 1, 512, 512])
Valeurs masque : tensor([0., 1.])


In [ ]:
!python src/train.py --mode teacher --epochs 10 --batch_size 5 --img_size 512


  Device : cuda
[train] 7908 images chargées
[val] 1694 images chargées
[test] 1696 images chargées

  MODE : TEACHER (U-Net++)
Traceback (most recent call last):
  File "/content/SegKD/src/train.py", line 327, in <module>
    main()
  File "/content/SegKD/src/train.py", line 314, in main
    train_teacher(args, device, loaders)
  File "/content/SegKD/src/train.py", line 54, in train_teacher
    logits, _ = model(images)
                ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1776, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1787, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/SegKD/src/models.py", line 48, in forward
    logits = self.model(x)
             ^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/t